# GC002 T1-FRESH-02 — 全新训练与完整评估

固定 Tesla T4 / FP32 / Seed 173020021 / Warmup 97 / 250,000 预测目标 / 977 更新。
每次启动使用新的输出目录，从随机初始化训练；不恢复模型、不创建 Checkpoint 或恢复包，结束后保存一次最终模型。
训练后明确执行完整 Validation、Development Holdout、生成、推理与显存评估；单项失败继续其他评估。
启用 GPU (Tesla T4)、Internet、GITHUB_TOKEN 和 KAGGLE_API_TOKEN Secrets，Save & Run All。数据仅从版本 142 下载，不访问 Independent Test。
总预算 5400 秒执行 + 600 秒发布 + 1200 秒余量。模型仅保存在本次 Kaggle Output；Git 只发布报告。


In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time

BRANCH = 'codex/gc002-t1-formal-readiness'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-gc002-t1-formal'
OUTPUT = WORKSPACE / 'epclm-gc002-t1-formal-bootstrap'
HARD_LIMIT_SECONDS = 7200
deadline = time.monotonic() + HARD_LIMIT_SECONDS

if not os.environ.get('KAGGLE_API_TOKEN', '').strip():
    from kaggle_secrets import UserSecretsClient
    os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN').strip()
if not os.environ['KAGGLE_API_TOKEN']:
    raise RuntimeError('Enable the existing Kaggle Secret KAGGLE_API_TOKEN.')

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix': os.killpg(process.pid, signal.SIGTERM)
        else: process.terminate()
        try: output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix': os.killpg(process.pid, signal.SIGKILL)
            else: process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Notebook time limit reached; recoverable output, if any, remains under ' + str(OUTPUT))
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Output is under ' + str(OUTPUT))

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'switch', '-C', BRANCH, 'FETCH_HEAD'])
    run_bounded(['git', '-C', str(REPO), 'clean', '-fd'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH,
        REPOSITORY, str(REPO)])

SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT = WORKSPACE / ('epclm-gc002-t1-formal-' + SOURCE_COMMIT[:12])

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before running T1 formal validation/training.')
run_bounded([sys.executable, str(REPO / 'research/projects/cellular-language-models/experiments/epclm-core-001/gc002-fair-architecture/t1_formal.py'),
    '--output', str(OUTPUT)], cwd=REPO, timeout_seconds=6000)
